# Embeddings

Um **embedding** representa um texto, uma imagem ou outro objeto como um vetor de $\mathbb{R}^d$, de modo que objetos de significado parecido fiquem próximos. Com os textos nessa forma, as técnicas das unidades anteriores se aplicam diretamente. Neste notebook simulamos um portal de notícias, com as notícias publicadas e o histórico de leitura dos usuários, e usamos embeddings em quatro aplicações de aprendizado não supervisionado: busca, modelagem de tópicos, detecção de anomalias e recomendação.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

plt.style.use('seaborn-v0_8-whitegrid')

A biblioteca `sentence-transformers` precisa ser instalada à parte com `pip install sentence-transformers`. Na primeira execução, os pesos do modelo são baixados e guardados em cache local.

## Fundamentação Matemática

Representar cada palavra por um vetor com um único 1 na sua posição do vocabulário, o *one-hot*, deixa todas as palavras à mesma distância umas das outras. Um embedding usa vetores densos, aprendidos por uma rede neural treinada em muito texto, em que a proximidade carrega significado. A proximidade é medida pela **similaridade de cosseno**

$$ \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|} $$

que vai de $-1$ a $1$. Com os vetores normalizados para norma 1, ela é simplesmente o produto interno $\mathbf{u} \cdot \mathbf{v}$.

## Dataset: Portal de Notícias

O portal tem 90 notícias, 15 de cada uma das seis seções, cada uma com a manchete e uma frase de complemento, e mais alguns textos que foram publicados por engano, como anúncios e testes. A seção de cada notícia serve apenas para avaliar os resultados e para analisar o comportamento dos leitores; a modelagem de tópicos e a detecção de anomalias não a utilizam.

In [ ]:
news = {
    'política': [
        "Câmara aprova em primeiro turno a proposta de reforma tributária. O texto segue agora para votação no Senado.",
        "Senado instala comissão para investigar contratos do governo federal. Os senadores terão 120 dias para concluir os trabalhos.",
        "Partidos negociam alianças para as eleições municipais do próximo ano. Os acordos envolvem candidaturas a prefeito nas capitais.",
        "Supremo Tribunal Federal julga ação sobre o foro privilegiado de parlamentares. O relator votou pela restrição do benefício.",
        "Deputados rejeitam pedido de urgência para o novo código eleitoral. A votação foi adiada para o próximo semestre legislativo.",
        "Ministro é convocado pela Câmara para explicar gastos com viagens oficiais. A oposição acusa o governo de falta de transparência.",
        "Tribunal Superior Eleitoral define regras para a propaganda dos candidatos nas redes sociais. Partidos que descumprirem as normas podem ser multados.",
        "Oposição protocola pedido de impeachment contra o governador do estado. A Assembleia Legislativa decidirá se aceita a denúncia.",
        "Congresso derruba veto do presidente ao projeto de desoneração da folha. Deputados e senadores votaram em sessão conjunta.",
        "Pesquisa eleitoral mostra empate técnico entre os dois principais candidatos à presidência. A margem de erro é de dois pontos percentuais.",
        "Vereadores aprovam reajuste do próprio salário em sessão tumultuada na Câmara Municipal. O prefeito ainda pode vetar a medida.",
        "Governo federal envia ao Congresso proposta de reforma administrativa. O texto muda as regras de estabilidade dos servidores públicos.",
        "Comissão de Constituição e Justiça aprova fim da reeleição para presidente, governadores e prefeitos. A proposta de emenda vai ao plenário.",
        "Justiça Eleitoral cassa o mandato de deputado por abuso de poder econômico na campanha. Ele ainda pode recorrer ao tribunal superior.",
        "Debate entre candidatos à prefeitura tem troca de acusações e poucas propostas. Os eleitores vão às urnas no próximo domingo.",
    ],
    'economia': [
        "Banco Central mantém a taxa Selic e sinaliza cautela com a inflação. A decisão do Copom foi unânime.",
        "Inflação oficial desacelera e fica abaixo do esperado pelo mercado. O IPCA acumulado em doze meses é o menor desde 2020.",
        "Dólar recua e bolsa de valores fecha em alta após dados de emprego nos Estados Unidos. O Ibovespa subiu mais de um por cento.",
        "Desemprego cai ao menor nível em dez anos, aponta pesquisa do IBGE. O número de trabalhadores com carteira assinada também cresceu.",
        "Produto Interno Bruto cresce acima das projeções no segundo trimestre. Agropecuária e serviços puxaram a alta da economia.",
        "Governo anuncia bloqueio de gastos no orçamento para cumprir a meta fiscal. O corte atinge investimentos de vários ministérios.",
        "Vendas no varejo têm o melhor resultado para o mês em dez anos. Supermercados e farmácias lideraram o crescimento do comércio.",
        "Endividamento das famílias atinge recorde, segundo a confederação do comércio. Cartão de crédito é a principal fonte das dívidas.",
        "Juros do cartão de crédito rotativo seguem acima de 400% ao ano. Economistas recomendam trocar a dívida por um empréstimo mais barato.",
        "Mercado eleva a previsão de crescimento da economia para este ano. O boletim Focus também projeta inflação menor.",
        "Produção da indústria cai pelo terceiro mês consecutivo. A alta dos juros encarece o crédito e reduz os investimentos das fábricas.",
        "Tesouro Direto bate recorde de investidores com a alta dos juros. Títulos atrelados à inflação são os mais procurados.",
        "Salário mínimo terá reajuste acima da inflação no próximo ano. O aumento eleva os gastos da Previdência com aposentadorias.",
        "Bancos ampliam a oferta de crédito imobiliário com juros menores. As taxas para financiamento da casa própria caíram pelo segundo mês.",
        "Preço da cesta básica cai em capitais após safra recorde de alimentos. Arroz, feijão e leite ficaram mais baratos no mês.",
    ],
    'esportes': [
        "Flamengo vence o clássico e assume a liderança do Campeonato Brasileiro. O gol da vitória saiu nos acréscimos do segundo tempo.",
        "Técnico da seleção brasileira convoca jogadores para as eliminatórias da Copa do Mundo. A lista tem três estreantes no ataque.",
        "Corinthians anuncia a contratação de atacante vindo do futebol europeu. O jogador assinou contrato por três temporadas.",
        "Palmeiras empata fora de casa e se complica na Libertadores. O time precisa vencer a última partida da fase de grupos.",
        "Técnico é demitido após sequência de cinco derrotas no campeonato. O clube ocupa a zona de rebaixamento da tabela.",
        "Atacante marca três gols e garante a classificação do time na Copa do Brasil. A torcida lotou o estádio para a partida de volta.",
        "Clube rebaixado anuncia reformulação completa do elenco para a disputa da Série B. Doze jogadores deixaram o time.",
        "Goleiro defende dois pênaltis e leva o time às semifinais do campeonato estadual. A decisão foi para as penalidades após empate sem gols.",
        "Confederação divulga a tabela do Campeonato Brasileiro do próximo ano. A primeira rodada tem dois clássicos regionais.",
        "Jogador sofre lesão no joelho e desfalca a equipe por seis meses. O meia vai passar por cirurgia ainda nesta semana.",
        "Seleção feminina vence a Argentina e se classifica para a final da Copa América. A artilheira marcou duas vezes no primeiro tempo.",
        "Zagueiro é expulso e time perde em casa na estreia do campeonato. O treinador reclamou da arbitragem após o jogo.",
        "Santos acerta a volta de ídolo para a próxima temporada. O meia-atacante foi revelado nas categorias de base do clube.",
        "Grêmio goleia o Internacional no Gre-Nal e encerra jejum de vitórias no clássico. O time tricolor marcou quatro gols no segundo tempo.",
        "Artilheiro do Brasileirão renova contrato e recusa proposta de clube árabe. Ele soma dezoito gols na competição.",
    ],
    'tecnologia': [
        "Nova versão de assistente de inteligência artificial passa a conversar por voz em português. O modelo de linguagem também entende imagens.",
        "Empresa lança smartphone com bateria que dura dois dias. O aparelho tem processador mais eficiente e câmera de 200 megapixels.",
        "Ataque hacker expõe dados de milhões de clientes de operadora de telefonia. Senhas e números de documentos vazaram na internet.",
        "Rede 5G chega a mais cem cidades do interior do país. As operadoras prometem internet móvel até dez vezes mais rápida.",
        "Pesquisadores desenvolvem chip que consome metade da energia dos processadores atuais. A tecnologia pode chegar aos computadores em três anos.",
        "Aplicativo de mensagens adiciona criptografia em chamadas de vídeo em grupo. A atualização já está disponível para Android e iPhone.",
        "Falha em servidores de computação em nuvem deixa sites e aplicativos fora do ar por horas. A empresa atribuiu o problema a um erro de software.",
        "Golpe com link falso enviado por mensagem rouba dados de usuários de celular. Especialistas em segurança digital recomendam não clicar em links desconhecidos.",
        "Universidade cria computador quântico experimental com cinquenta qubits. O equipamento será usado em pesquisas de criptografia.",
        "Rede social muda o algoritmo e reduz o alcance de publicações com links externos. Criadores de conteúdo reclamam da mudança.",
        "Modelo de inteligência artificial é treinado para programar e corrigir erros em código. A ferramenta já é usada por desenvolvedores de software.",
        "Lei de proteção de dados aplica primeira multa milionária a empresa de tecnologia. A empresa compartilhou dados pessoais de usuários sem consentimento.",
        "Fabricante apresenta notebook com tela dobrável e teclado sem fio. O computador roda o novo sistema operacional da empresa.",
        "Navegador de internet ganha recurso de inteligência artificial que resume páginas. A função funciona em computadores e celulares.",
        "Empresa de chips anuncia fábrica de semicondutores no país. A produção de processadores deve começar em 2027.",
    ],
    'saúde': [
        "Ministério da Saúde amplia a vacinação contra a gripe para toda a população. As doses estão disponíveis nos postos de saúde.",
        "Casos de dengue crescem e estados decretam situação de emergência. Os hospitais reforçam o atendimento a pacientes com febre.",
        "Estudo mostra que caminhar trinta minutos por dia reduz o risco de infarto. Os pesquisadores acompanharam dez mil pacientes.",
        "Anvisa aprova novo medicamento para o tratamento de diabetes tipo 2. O remédio ajuda a controlar a glicose no sangue.",
        "Hospitais públicos enfrentam falta de médicos nas emergências. Pacientes esperam mais de oito horas por atendimento.",
        "SUS passa a oferecer tratamento gratuito para enxaqueca crônica. O medicamento será distribuído em centros de referência.",
        "Especialistas alertam para o aumento de casos de ansiedade e depressão entre jovens. Psiquiatras recomendam procurar ajuda cedo.",
        "Campanha de vacinação contra o sarampo tem baixa adesão nas capitais. Médicos alertam para o risco de surto da doença.",
        "Novo exame de sangue detecta câncer de intestino em estágio inicial. O diagnóstico precoce aumenta as chances de cura do paciente.",
        "Dormir menos de seis horas por noite eleva o risco de doenças cardíacas. O estudo foi publicado em revista de cardiologia.",
        "Farmácia Popular amplia a lista de remédios distribuídos de graça. Medicamentos para hipertensão e asma estão incluídos.",
        "Vacina contra a dengue começa a ser aplicada em crianças e adolescentes. São necessárias duas doses com intervalo de três meses.",
        "Fila para cirurgias no SUS diminui após mutirão nos hospitais. Foram realizadas mais de cinco mil operações em um mês.",
        "Casos de tuberculose voltam a crescer nas grandes cidades. A doença tem tratamento gratuito, mas exige seis meses de remédios.",
        "Pediatras recomendam limitar o tempo de tela para proteger o sono e a saúde mental das crianças. O uso excessivo de celular causa insônia.",
    ],
    'cultura': [
        "Filme brasileiro é indicado ao Oscar de melhor filme internacional. O longa conta a história de uma família durante a ditadura.",
        "Festival de cinema de Gramado anuncia os filmes da mostra competitiva. Diretores estreantes dominam a seleção deste ano.",
        "Bienal do Livro reúne escritores e recebe meio milhão de visitantes. Os lançamentos de romances e poesia lotaram os estandes.",
        "Museu reabre após reforma com exposição de arte indígena. As obras incluem pinturas, esculturas e peças de cerâmica.",
        "Escritora vence o Prêmio Jabuti com romance sobre migração no Nordeste. O livro é o terceiro da autora.",
        "Peça de teatro estreia em São Paulo após temporada de sucesso no Rio. O espetáculo é uma adaptação de clássico da literatura.",
        "Exposição de fotografia retrata o cotidiano das periferias brasileiras. As imagens ficam em cartaz na galeria até março.",
        "Cantor lança álbum com participações de músicos de cinco países. O disco mistura samba, jazz e música eletrônica.",
        "Orquestra sinfônica faz concerto gratuito no parque no fim de semana. O programa inclui obras de Villa-Lobos e Beethoven.",
        "Documentário sobre a Amazônia é premiado em festival internacional de cinema. O diretor filmou durante dois anos na floresta.",
        "Artista plástico inaugura mural gigante no centro da cidade. A pintura homenageia poetas e escritores locais.",
        "Escola de samba campeã do carnaval anuncia o enredo do próximo desfile. O tema será a história da música popular brasileira.",
        "Romance clássico de Machado de Assis ganha nova edição ilustrada. O livro traz ensaios de críticos literários.",
        "Cinemas recebem mostra com filmes restaurados do cinema novo. As sessões acontecem em dez capitais.",
        "Banda de rock anuncia turnê de despedida com shows em oito cidades. Os ingressos para o primeiro show esgotaram em uma hora.",
    ],
}
others = [
    "Vendo sofá de três lugares em ótimo estado, tratar pelo telefone.",
    "Receita de bolo de cenoura com cobertura de chocolate em dez minutos.",
    "Horóscopo do dia: arianos terão sorte no amor e devem evitar discussões.",
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit.",
    "teste teste 123 publicação de teste favor ignorar",
]

texts = [t for items in news.values() for t in items] + others
sections = np.array([s for s, items in news.items() for _ in items] + ['outros'] * len(others))
print(f"{len(texts)} textos")

Os leitores são simulados. Cada um tem duas seções preferidas, sorteadas entre alguns perfis comuns, como quem acompanha política e economia, e lê de 6 a 12 notícias, quase todas das seções preferidas e algumas ao acaso. O resultado é uma tabela com uma linha por leitura.

In [ ]:
rng = np.random.default_rng(0)
profiles = [('política', 'economia'), ('economia', 'tecnologia'), ('esportes', 'cultura'),
            ('saúde', 'cultura'), ('tecnologia', 'cultura'), ('esportes', 'saúde')]
section_names = list(news)

reads = []
for user in range(300):
    favorites = profiles[rng.integers(len(profiles))]
    for _ in range(rng.integers(6, 13)):
        section = rng.choice(favorites) if rng.random() < 0.85 else rng.choice(section_names)
        reads.append((user, rng.choice(np.where(sections == section)[0])))

reads = pd.DataFrame(reads, columns=['usuário', 'notícia']).drop_duplicates()
reads.head()

## Gerando Embeddings

Usamos o `paraphrase-multilingual-MiniLM-L12-v2`, um modelo pré-treinado que produz vetores de 384 dimensões e entende mais de 50 idiomas, incluindo o português.

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')


def embed(texts):
    return model.encode(texts, normalize_embeddings=True)


X = embed(texts)
print(f"X: {X.shape}")

## Aplicação: Busca Semântica

A consulta vira um vetor, e as notícias são ordenadas pela similaridade com ela. Isso encontra notícias de significado parecido mesmo sem palavras em comum com a consulta.

In [ ]:
def search(query, k=3):
    scores = X @ embed([query])[0]
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({'similaridade': scores[top], 'seção': sections[top],
                         'notícia': [texts[i] for i in top]}).round(3)


search("mosquito transmissor de doenças")

## Aplicação: Modelagem de Tópicos

A **modelagem de tópicos** descobre os assuntos de uma coleção de textos sem rótulos, agrupando os embeddings. Usamos o DBSCAN, que não exige escolher o número de grupos e marca como ruído os textos que não pertencem a nenhum. Como densidades em 384 dimensões são pouco informativas, primeiro reduzimos os embeddings para 5 dimensões com o UMAP, com `min_dist=0`, que deixa os grupos compactos. Essa é a receita de ferramentas como o BERTopic. O `eps` é escolhido pelo gráfico das distâncias até o 4º vizinho, como na aula de DBSCAN.

In [ ]:
Z = umap.UMAP(n_components=5, n_neighbors=10, min_dist=0.0, metric='cosine', random_state=42).fit_transform(X)

k_distances = NearestNeighbors(n_neighbors=4).fit(Z).kneighbors(Z)[0][:, -1]
plt.figure(figsize=(8, 4))
plt.plot(np.sort(k_distances))
plt.axhline(0.4, color='gray', ls='--')
plt.xlabel('textos ordenados')
plt.ylabel('distância ao 4º vizinho')
plt.show()

A curva sobe rápido no final, nos textos mais isolados, e usamos `eps=0.4`, logo antes desse ponto. Para dar nome a cada tópico, mostramos as duas notícias mais próximas do centro do grupo.

In [ ]:
topics = DBSCAN(eps=0.4, min_samples=4).fit_predict(Z)

rows = []
for t in range(topics.max() + 1):
    members = np.where(topics == t)[0]
    center = X[members].mean(axis=0)
    closest = members[np.argsort(X[members] @ center)[::-1][:2]]
    rows.append({'tópico': t, 'notícias': len(members), 'mais representativas': ' | '.join(texts[i] for i in closest)})

print(f"Tópicos: {topics.max() + 1}, textos classificados como ruído: {np.sum(topics == -1)}")
pd.DataFrame(rows)

Comparando os tópicos encontrados com as seções do portal:

In [ ]:
noise = topics == -1
print(f"ARI: {adjusted_rand_score(sections[~noise], topics[~noise]):.3f}")
pd.crosstab(topics, sections)

Cada tópico corresponde a uma seção, sem que nenhuma seção tenha sido usada no processo. Para visualizar, projetamos os embeddings em 2D com o UMAP, com o ruído em cinza.

In [ ]:
Z_2d = umap.UMAP(n_components=2, n_neighbors=10, metric='cosine', random_state=42).fit_transform(X)

plt.figure(figsize=(10, 7))
plt.scatter(Z_2d[noise, 0], Z_2d[noise, 1], s=60, color='lightgray', label='ruído')
plt.scatter(Z_2d[~noise, 0], Z_2d[~noise, 1], s=40, c=topics[~noise], cmap='tab10')
for t in range(topics.max() + 1):
    plt.text(*np.median(Z_2d[topics == t], axis=0), str(t), fontsize=14, weight='bold')
plt.xticks([])
plt.yticks([])
plt.legend()
plt.show()

## Aplicação: Detecção de Anomalias

O ruído do DBSCAN já é uma forma de detecção de anomalias: são os textos em regiões de baixa densidade, que não se parecem o bastante com nenhum grupo. Vejamos quais foram.

In [ ]:
pd.DataFrame({'seção': sections[noise], 'texto': np.array(texts)[noise]})

O ruído reúne todos os textos que não são notícias, junto com algumas notícias de fronteira, que ficam entre dois assuntos. Numa coleção real, examinar o ruído é a forma de encontrar spam, conteúdo fora do escopo ou problemas de qualidade nos dados.

## Aplicação: Recomendação

Para recomendar notícias a um leitor, representamos o seu **perfil** pela média dos embeddings das notícias que ele leu, normalizada para norma 1. As recomendações são as notícias mais próximas do perfil que ele ainda não leu.

In [ ]:
def profile(user):
    read = reads.loc[reads['usuário'] == user, 'notícia']
    p = X[read].mean(axis=0)
    return p / np.linalg.norm(p), set(read)


def recommend(user, k=5):
    p, read = profile(user)
    scores = X @ p
    scores[list(read)] = -np.inf   # não recomenda o que já foi lido
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({'similaridade': scores[top], 'seção': sections[top],
                         'notícia': [texts[i] for i in top]}).round(3)


user = 8
print(reads.loc[reads['usuário'] == user, 'notícia'].map(lambda i: sections[i]).value_counts().to_string())
recommend(user)

As recomendações vêm das seções que o leitor mais lê, sem repetir notícias já lidas. Para medir a qualidade, escondemos a última notícia lida por cada leitor, montamos o perfil com as demais e verificamos se a notícia escondida aparece entre as 10 recomendações. Comparamos com recomendar 10 notícias ao acaso.

In [ ]:
hits = []
for user, group in reads.groupby('usuário'):
    hidden = group['notícia'].iloc[-1]
    seen = group['notícia'].iloc[:-1]
    p = X[seen].mean(axis=0)
    scores = X @ (p / np.linalg.norm(p))
    scores[seen] = -np.inf
    hits.append(hidden in np.argsort(scores)[::-1][:10])

print(f"Acerto do perfil: {np.mean(hits):.1%}")
print(f"Acerto ao acaso: {10 / (len(texts) - reads.groupby('usuário').size().mean()):.1%}")

O perfil acerta bem mais que o acaso, usando só o conteúdo das notícias. Também podemos olhar para os leitores em conjunto e perguntar **quem lê uma seção também lê qual outra**. Montamos uma tabela com uma linha por leitor e uma coluna por seção, com o número de notícias lidas, e medimos a similaridade de cosseno entre as colunas: duas seções são parecidas quando são lidas pelas mesmas pessoas.

In [ ]:
counts = pd.crosstab(reads['usuário'], sections[reads['notícia']])
C = counts.to_numpy(dtype=float)
C /= np.linalg.norm(C, axis=0)             # normaliza cada coluna (seção)
similarity = pd.DataFrame(C.T @ C, index=counts.columns, columns=counts.columns)

plt.figure(figsize=(7, 6))
plt.imshow(similarity, cmap='viridis')
plt.colorbar(label='similaridade de cosseno')
plt.xticks(range(len(similarity)), similarity.columns, rotation=45)
plt.yticks(range(len(similarity)), similarity.columns)
plt.grid(False)
plt.show()

Os pares mais claros, fora da diagonal, são as seções lidas pelas mesmas pessoas, como política e economia. Essa é a ideia por trás de recomendações do tipo "quem leu isto também leu", que usam apenas o comportamento dos leitores, sem olhar o conteúdo, e complementam a recomendação por embeddings.

## Exercícios

### Exercício 1: Parâmetros do DBSCAN

Refaça a modelagem de tópicos com `eps` igual a 0,3, 0,5 e 0,6 e com `min_samples` igual a 3 e 6. Para cada combinação, reporte o número de tópicos, o ARI fora do ruído e quantos dos textos que não são notícias foram marcados como ruído. Existe uma combinação que acerte bem os tópicos e as anomalias ao mesmo tempo? Depois, aplique o DBSCAN diretamente aos embeddings de 384 dimensões, com `metric='cosine'`, e explique a diferença.

In [ ]:
# Seu código aqui

### Exercício 2: Leitores Anômalos

Acrescente à tabela `reads` 10 leitores "robôs", que leem 30 notícias sorteadas ao acaso, sem preferência por seção. Represente cada leitor pela proporção de leituras em cada seção e use o DBSCAN para encontrar os leitores anômalos. Os robôs são encontrados? Algum leitor normal é marcado como anômalo?

In [ ]:
# Seu código aqui

### Exercício 3: Recomendação para Leitores Novos

Um leitor novo ainda não leu nenhuma notícia, então não tem perfil. Escreva uma função que receba um texto com os interesses do leitor, como "gosto de futebol e de filmes", e recomende as notícias mais próximas dele. Depois, compare as recomendações por perfil de 5 leitores com as notícias mais lidas do portal, a recomendação mais simples possível. Em que situações cada abordagem é melhor?

In [ ]:
# Seu código aqui